# 第6回　確率分布と標本／中心極限定理
## ―― なぜ「一部」から「全体」を語れるのか

統計学Ⅰ（B）　／　北星学園大学

今日から **推測統計**。手元のデータ（標本）から、見ていない全体（母集団）を推し量る。注目は ――

> 全部見なくていい。**標本の平均は、母集団の平均の周りに、きれいな形で散らばる。**

ただし今日は、**その「きれいな形」がなかなか現れない場合**も見ることになる。

In [ ]:
# 準備：ライブラリと、霊長類376種のデータを読み込む。▶ を押すだけ。
!pip install -q japanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401
from scipy import stats

def _build_from_source():
    """公開データ（PanTHERIA）から、この授業で使う形に組み立て直す。"""
    # 原典（PanTHERIA）から組み立て直す。まずリポジトリ同梱の複製、だめなら発行元から。
    # 発行元は User-Agent を見て弾くことがあるため、明示して取得する。
    import io, urllib.request
    SRCS = [
        "https://raw.githubusercontent.com/aonoa68/toukei-1/main/docs/data/PanTHERIA_1-0_WR05_Aug2008.txt.gz",
        "https://esapubs.org/archive/ecol/E090/184/PanTHERIA_1-0_WR05_Aug2008.txt",
    ]
    fam = {"Cercopithecidae":"オナガザル科","Cebidae":"オマキザル科","Pitheciidae":"サキ科",
           "Atelidae":"クモザル科","Cheirogaleidae":"コビトキツネザル科","Lemuridae":"キツネザル科",
           "Galagidae":"ガラゴ科","Hylobatidae":"テナガザル科","Indriidae":"インドリ科",
           "Lorisidae":"ロリス科","Lepilemuridae":"イタチキツネザル科","Aotidae":"ヨザル科",
           "Hominidae":"ヒト科","Tarsiidae":"メガネザル科","Daubentoniidae":"アイアイ科"}
    cols = {"MSW05_Binomial":"学名","MSW05_Genus":"属","5-1_AdultBodyMass_g":"体重g",
            "13-1_AdultHeadBodyLen_mm":"頭胴長mm","5-3_NeonateBodyMass_g":"新生児体重g",
            "10-2_SocialGrpSize":"集団サイズ","9-1_GestationLen_d":"妊娠期間日",
            "25-1_WeaningAge_d":"離乳日齢","3-1_AgeatFirstBirth_d":"初産日齢",
            "14-1_InterbirthInterval_d":"出産間隔日","15-1_LitterSize":"一腹産子数",
            "17-1_MaxLongevity_m":"最長寿命月","22-1_HomeRange_km2":"行動圏km2",
            "21-1_PopulationDensity_n/km2":"個体群密度","26-1_GR_Area_km2":"分布域km2",
            "6-2_TrophicLevel":"栄養段階","12-1_HabitatBreadth":"生息環境幅",
            "28-2_Temp_Mean_01degC":"平均気温01","28-1_Precip_Mean_mm":"月降水量mm"}
    src = None
    for _url in SRCS:
        try:
            _req = urllib.request.Request(_url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(_req, timeout=60) as _r:
                _raw = _r.read()
            _comp = "gzip" if _url.endswith(".gz") else None
            src = pd.read_csv(io.BytesIO(_raw), sep="\t", compression=_comp)
            break
        except Exception:
            continue
    if src is None:
        raise RuntimeError("原典データを取得できませんでした")
    p = src[src["MSW05_Order"] == "Primates"]
    out = p[list(cols)].rename(columns=cols)
    out.insert(1, "科", p["MSW05_Family"].map(fam))
    t = out.pop("平均気温01")
    out["平均気温C"] = np.where(t == -999, -999, (t / 10).round(1))
    return out.replace(-999, np.nan).sort_values("学名").reset_index(drop=True)

try:
    df = pd.read_csv("https://aonoa68.github.io/toukei-1/data/primates.csv")
except Exception:
    df = _build_from_source()

print("種数:", len(df), " 科数:", df["科"].nunique())
df.head()

---
## フック：なぜ「一部」で「全体」が分かる？

- 視聴率は、全国数千万世帯のうち**たった数百世帯**を調べて出している。
- 味噌汁の味見は、鍋全体を飲まなくても**ひと匙**で分かる。

なぜ一部で全体を語れるのか。今日はその数学的な裏づけ＝**中心極限定理**を、自分の目で見る。

---
## 1. 母集団と標本

- **母集団**：本当に知りたい全体
- **標本**：実際に調べる一部

今日は「**体重の記録がある霊長類265種**」を母集団とみなす。まず母集団の形を見よう。
第2回で見たとおり、**右に強く歪んでいて、正規分布ではない。**

In [ ]:
rng = np.random.default_rng(2026)

母集団 = df["体重g"].dropna().values
mu = 母集団.mean()
sigma = 母集団.std()

print(f"母集団の大きさ N = {len(母集団)} 種")
print(f"母平均 μ = {mu:,.0f} g")
print(f"母標準偏差 σ = {sigma:,.0f} g")
print(f"歪度 = {stats.skew(母集団):.2f}   ← 0から大きく離れている＝強く歪んでいる")

plt.figure(figsize=(7,3.5))
plt.hist(母集団, bins=40, color="#80cbc4", edgecolor="white")
plt.axvline(mu, color="#1565c0", lw=2, label=f"母平均 {mu:,.0f}g")
plt.xlabel("体重（g）"); plt.ylabel("種数"); plt.legend()
plt.title("母集団は正規分布ではない（右に強く歪む）")
plt.show()

> **歪度 7.60。** 第3回で扱った通り、これはかなり極端な歪みである。（比較のため：一般的な「歪んでいる」データは歪度1〜2程度）

**この母集団で中心極限定理が本当に効くのか**を、今日は確かめることになる。

---
## 2. 標本を1つ取ってみる（無作為抽出）

母集団から **n=30種** をランダムに選び、その標本平均を出す。実行するたびに少しずつ違う値になる（運で変わる）。

In [ ]:
標本 = rng.choice(母集団, size=30, replace=False)
print("選ばれた30種の体重(g):")
print(np.round(np.sort(標本)).astype(int))
print()
print(f"この標本の平均 = {標本.mean():,.0f} g （母平均 {mu:,.0f} に近いが、ぴったりではない）")

> 選ばれた30種を眺めてほしい。**数十gの種と数万gの種が混ざっている。**
> 大型種が1種でも多く入れば標本平均は跳ね上がり、入らなければ低く出る。
> **1回の標本は、運に大きく左右される。**

---
## 3. 標本平均を「何度も」取ると…（標本分布）

1回だけだと運に左右される。では **n=30 の標本取り→平均を2000回くりかえす**と、標本平均たちはどんな形に散らばる？

母集団は強く歪んでいた。標本平均の分布も歪むだろうか？

In [ ]:
標本平均たち = np.array([rng.choice(母集団, 30).mean() for _ in range(2000)])

print(f"標本平均の平均       = {標本平均たち.mean():,.0f} g （母平均 {mu:,.0f} とほぼ一致）")
print(f"標本平均のばらつき(SD) = {標本平均たち.std():,.0f} g")
print(f"標本平均の分布の歪度   = {stats.skew(標本平均たち):.2f}")
print(f"（母集団の歪度は {stats.skew(母集団):.2f} だった → かなり小さくなった）")

plt.figure(figsize=(7,3.5))
plt.hist(標本平均たち, bins=40, color="#e8503a", edgecolor="white")
plt.axvline(mu, color="#1565c0", lw=2, label=f"母平均 {mu:,.0f}")
plt.xlabel("標本平均（n=30）"); plt.ylabel("回数"); plt.legend()
plt.title("母集団の歪度7.6 → 標本平均の分布の歪度1.3 へ")
plt.show()

**歪度が 7.60 → 約1.3 まで下がった。** 確かに釣鐘型に近づいている。これが **中心極限定理(CLT)** である。

> 母集団がどんな形でも、標本平均をたくさん集めると、その分布は**正規分布に近づく**。

### ただし、まだ正規分布ではない

よく「**n が30あれば正規分布とみなしてよい**」と言われる。しかし今のグラフをよく見てほしい。**右にまだ裾を引いている。**歪度1.3は、決して0ではない。

**「n=30ルール」は、母集団がそこそこ対称なときの目安にすぎない。**この母集団のように極端に歪んでいると、30では足りない。

どれくらい必要なのか、実際に増やして確かめよう。

---
## 4. 標本サイズ n を変えると ―― 分布が「締まる」

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(15, 3.3))
print("  n     標本平均SD    σ/√n      歪度")
print("-" * 42)
for i, n in enumerate([5, 30, 100, 500]):
    means = np.array([rng.choice(母集団, n).mean() for _ in range(2000)])
    ax[i].hist(means, bins=30, color="#80cbc4", edgecolor="white")
    ax[i].axvline(mu, color="#1565c0", lw=2)
    ax[i].set_title(f"n={n}\n歪度={stats.skew(means):.2f}")
    ax[i].set_xlabel("標本平均")
    print(f"{n:>4}  {means.std():>9,.0f}  {sigma/np.sqrt(n):>9,.0f}  {stats.skew(means):>8.2f}")

plt.suptitle("n が大きいほど、母平均の近くに締まり、左右対称に近づく")
plt.tight_layout(); plt.show()

2つのことが同時に起きている。

**① ばらつきが小さくなる（締まる）**

標本平均のばらつき（**標準誤差 SE**）は **σ/√n** にぴったり一致している。
n を4倍にするとSEは半分。だから「**精度を2倍にするには4倍のデータが要る**」。

$$ 標準誤差\ SE = \frac{母標準偏差\ \sigma}{\sqrt{n}} $$

**② 形が正規分布に近づく（ゆっくり）**

| n | 歪度 |
|---:|---:|
| 母集団 | 7.60 |
| 5 | 約 3.5 |
| 30 | 約 1.3 |
| 100 | 約 0.7 |
| 500 | 約 0.3 |

**ばらつきは 1/√n できれいに縮むのに、歪みはなかなか取れない。**
n=500 でもまだ0.3残っている。

> **CLTは「いつかは正規になる」と言っているだけで、「n=30で正規になる」とは言っていない。**
> 必要な n は、母集団がどれだけ歪んでいるかで決まる。

---
## 5. 尺度を選ぶと、必要な n が変わる

第3回で、体重は**対数にすると左右対称に近づいた**（歪度 7.60 → −0.40）。
対数の体重を母集団にしたら、CLTはどれくらいで効くだろうか。

In [ ]:
対数母集団 = np.log10(母集団)
print(f"対数母集団の歪度 = {stats.skew(対数母集団):.2f}")
print()
print("  n     歪度（対数）   歪度（生）")
print("-" * 34)
for n in [5, 30]:
    m_log = np.array([rng.choice(対数母集団, n).mean() for _ in range(2000)])
    m_raw = np.array([rng.choice(母集団, n).mean() for _ in range(2000)])
    print(f"{n:>4}      {stats.skew(m_log):>6.2f}      {stats.skew(m_raw):>6.2f}")

**対数にすると、n=5 の時点でもう歪度は −0.2 程度。** 生のスケールで n=500 かけても届かなかった水準である。

> **同じデータ、同じ標本サイズ。違うのは尺度だけ。**
> どの尺度で分析するかは、必要なデータ量を左右する実務的な判断でもある。

（ただし注意：対数の平均を戻しても、元の平均にはならない。対数で分析すると「幾何平均」の話になる。何を推定したいのかで選ぶこと。）

---
## 6. 大事な区別：「標本そのもの」≠「標本平均の分布」

よくある誤解：「標本を取ると正規分布になる」。**ちがう。**正規に近づくのは**標本平均の分布**であって、標本そのものは母集団と同じ形（歪んだまま）だ。並べて確認しよう。

In [ ]:
一つの標本 = rng.choice(母集団, 1000)                       # 標本そのもの（1000回抽出）
標本平均の分布 = np.array([rng.choice(母集団, 30).mean() for _ in range(2000)])

fig, ax = plt.subplots(1, 2, figsize=(11, 3.3))
ax[0].hist(一つの標本, bins=40, color="#80cbc4", edgecolor="white")
ax[0].set_title(f"標本そのもの → 歪んだまま（歪度 {stats.skew(一つの標本):.2f}）")
ax[0].set_xlabel("体重(g)")
ax[1].hist(標本平均の分布, bins=30, color="#e8503a", edgecolor="white")
ax[1].set_title(f"標本平均(n=30)の分布（歪度 {stats.skew(標本平均の分布):.2f}）")
ax[1].set_xlabel("標本平均")
plt.tight_layout(); plt.show()

---
## 7. ただし「無作為（ランダム）」が大前提

CLTが効くのは、標本が**母集団から偏りなく・独立に**選ばれているとき。

- もし「声の大きい人」「答えやすい人」ばかり選ぶと、いくら数を増やしても**偏ったまま**。これは数を増やしても直らない（標本数の問題ではなく、抽出の偏りの問題）。
- みんなが**互いに影響し合って**答えをそろえる（＝空気を読む）と、標本の**独立性**が壊れ、推測の前提が崩れる。

> ⚠️ **数の多さは、偏りを直さない**
> 
> 偏った集め方をしたデータは、何万件あっても母集団を正しく代表しない。**『たくさん集めた』は『正しく集めた』ではない。**

### 今日の「母集団」自体が、実は偏った標本である

ここまで265種を母集団として扱ってきた。しかし ――

**この265種は「体重が測られた種」でしかない。**地球上の霊長類376種のうち、体重の記録があるのは70%。残り111種は測られていない。

そして、測られている種には偏りがある。科ごとに見てみよう。

In [ ]:
tbl = pd.DataFrame({
    "全種数": df["科"].value_counts(),
    "体重の記録あり": df.dropna(subset=["体重g"])["科"].value_counts(),
})
tbl["記録率%"] = (tbl["体重の記録あり"] / tbl["全種数"] * 100).round(1)
tbl.sort_values("全種数", ascending=False)

**科によって記録率が大きく違う。** よく研究されてきた科は埋まり、研究者が少ない・観察が難しい科は空欄が多い。

つまり「霊長類の平均体重 5,881g」という今日の母平均は、**厳密には「よく研究されてきた霊長類の平均体重」**である。

> **データの形は、世界の形であると同時に、研究の歴史の形でもある。**
> どんなに立派な統計手法を使っても、**何が測られなかったか**は数字の中に現れない。
> それに気づけるのは、対象を知っている人間だけである。

---
## 今日のまとめ

| 概念 | ひとこと |
|---|---|
| 母集団 / 標本 | 知りたい全体 / 実際に調べる一部 |
| 中心極限定理(CLT) | 母集団がどんな形でも、標本平均の分布は正規に**近づく** |
| **n=30ルールの限界** | **母集団が強く歪んでいると30では足りない**（歪度7.6→n=30でまだ1.3） |
| 標準誤差 SE = σ/√n | 標本平均のばらつき。n を4倍にするとSEは半分 |
| 標本 ≠ 標本平均 | 正規に近づくのは「標本平均の分布」。標本そのものは母集団の形のまま |
| 無作為抽出 | CLTの大前提。偏った抽出は数を増やしても直らない |

> **一部から全体を語れるのは、標本平均がCLTによって正規分布に近づくから。**
> ただし「無作為に集めた」ことが大前提 ―― 数の多さは偏りを直さない。

そして今日の裏テーマ ――

> **教科書に書いてある目安（n=30）を、自分のデータで確かめた。そして通用しなかった。**
> 目安は目安であって、法則ではない。**確かめる手段を、あなたはもう持っている。**

次回からは、この標本平均の散らばり（SE）を使って、母平均を**区間**で推定していく。

**課題（Moodle）**：CLTのシミュレーション結果を読み、「n を増やすと標本平均の分布に何が起きたか」を、ばらつきと形の2点に分けて説明する。

---

!!! quote "このデータの出典"
    Jones, K.E. et al. (2009) PanTHERIA: a species-level database of life history,
    ecology, and geography of extant and recently extinct mammals.
    *Ecology* 90(9): 2648. Ecological Archives E090-184.

    霊長類376種の行だけを抜き出し、列を選び、気温の単位を直したもの。値は変えていない。